# p4 · Relation 추론 (concept 쌍 → Relation Type)

p3를 통과한 concept(=Object Type) 사이의 **Relation Type** 초안을 원문 청크에서 추론한다. is-a(계층)는 p3의 몫이라 여기서 다루지 않는다.

**Gate 4**
1. domain/range가 실제 있는 concept인가 (닫힌 어휘) → 아니면 p3로 (concept 누락)
2. 방향과 다중성: 형식은 스키마로, 청크 간 다중성 충돌은 사람 큐, 방향은 judge가 본다
3. 근거 문장: 원문에 그대로 있고, domain과 range의 표기가 문장에 나오는가 → 아니면 p1로 (term 누락 · 청크 경계)
4. confidence가 낮으면 커밋하지 않고 보류 (사람 큐)

L1 결정론 → L3 judge(독립 호출, relation 한 건씩) → 보류. 관계 지표는 따로 보고한다(pro-team 반례: 통합 점수 뒤에 관계 0건이 숨었다).

참고: GrOIL(domain/range 닫힌 어휘, 낮은 confidence 보류), RIGOR(Gen-LLM + 독립 Judge-LLM + 결정론 검증 후에만 병합).

입력: `outputs/p3/concepts.json`, `outputs/p2/normed_terms.json`, 원문 → 출력: `outputs/p4/`

## 설정

In [1]:
import re
from collections import Counter, defaultdict
from typing import Literal

import spacy
from pydantic import BaseModel, Field

from common import GEMINI_MODEL, OUTPUT_ROOT, gate_record, generate_json, load_chunks, read_json, stage_dir, write_json

P3 = read_json(OUTPUT_ROOT / "p3" / "concepts.json")
P2 = read_json(OUTPUT_ROOT / "p2" / "normed_terms.json")
OUTPUT_DIR = stage_dir("p4")

CONF_THRESHOLD = 0.7   # 생성 confidence와 judge confidence 중 작은 값이 이보다 낮으면 보류

# True면 저장된 relation 후보(proposals.json)와 judge 판정(judge_cache.json)을 재사용한다.
# 게이트 규칙만 바꿔 다시 돌릴 때 LLM 출력의 흔들림 없이 규칙 변경 효과만 본다.
REUSE_CACHE = True
PROPOSALS_PATH = OUTPUT_DIR / "proposals.json"
JUDGE_CACHE_PATH = OUTPUT_DIR / "judge_cache.json"

doc_id, doc_text, chunks = load_chunks()
chunk_by_id = {c["chunk_id"]: c for c in chunks}
concepts = {c["name"]: c for c in P3["concepts"]}
normed = {n["normed_id"]: n for n in P2["normed_terms"]}
print(GEMINI_MODEL, "concepts:", len(concepts), "chunks:", len(chunks))

gemini-3.5-flash concepts: 17 chunks: 11


## concept 표기 사전

근거 문장 검사(Gate 4-③)에 쓴다. concept 이름을 띄어 쓴 형태, 이름의 머리명사(예: RecordedEvent → event), evidence·instance term의 표면 변형.

비교는 spaCy 표제어(lemma)로 한다. 문장과 표기를 모두 토큰화하고, 표기의 토큰열이 문장 안에 연속으로 나오면 일치다. 토큰끼리는 원형(소문자)이나 표제어 중 하나만 같아도 같다고 본다. 예: `activities` ↔ `activity`, `recorded` ↔ `record`. 짧은 표기는 문맥 없이 태깅돼 표제어가 달라질 수 있어서 원형 비교를 함께 둔다.

In [2]:
def norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[-/]", " ", s.lower())).strip()


def split_camel(name: str) -> str:
    return re.sub(r"(?<!^)(?=[A-Z])", " ", name).lower()


lexicon = {}
for name, c in concepts.items():
    forms = {split_camel(name), split_camel(name).split()[-1]}
    for i in c["evidence_term_ids"] + c["instance_term_ids"]:
        forms.update(normed[i]["surface_variants"])
    lexicon[name] = sorted({norm_text(f) for f in forms}, key=len, reverse=True)


nlp = spacy.load("en_core_web_sm", exclude=["parser", "ner"])


def tokens(text: str) -> list[tuple[str, str]]:
    """(원형 소문자, 표제어 소문자) 목록. 구두점 제외."""
    return [(t.lower_, t.lemma_.lower()) for t in nlp(norm_text(text)) if not t.is_punct and not t.is_space]


lexicon_tokens = {name: [tokens(f) for f in forms] for name, forms in lexicon.items()}


def same(a: tuple[str, str], b: tuple[str, str]) -> bool:
    return a[0] == b[0] or a[1] == b[1]


def mentions_concept(sentence: str, name: str) -> bool:
    s = tokens(sentence)
    for f in lexicon_tokens[name]:
        n = len(f)
        if n and any(all(same(s[i + k], f[k]) for k in range(n)) for i in range(len(s) - n + 1)):
            return True
    return False


for name in list(concepts)[:5]:
    print(f"{name:<22} {lexicon[name][:5]}")
print(mentions_concept("They are meaningful only where their endpoint activities are available.", "ProcessActivity"))

BusinessRule           ['invoice before goods receipt three way matching', 'ordering, cardinality, and exclusiveness rules', 'invoice after goods receipt three way matching', 'contractual service level thresholds', 'invoice matching procedure']
Case                   ['consignment case', 'item cases', 'case unit', 'item case', 'case']
Company                ['multinational paints and coatings company', 'business entity', 'subsidiary', 'company']
EventLog               ['consignment log', 'available log', 'event log', 'log']
Goods                  ['goods']
True


## 제안: 청크마다 relation 후보

concept 목록(닫힌 어휘)과 청크 원문을 주고, 청크에 근거가 있는 relation만 제안하게 한다. 다중성은 domain 기준이다 (`1:N` = domain 하나에 range 여럿).

In [3]:
class RelationProposal(BaseModel):
    name: str = Field(description="lowerCamelCase verb phrase read from domain to range, e.g. hasItem, suppliedBy.")
    domain: str = Field(description="Concept name, exactly as listed.")
    range: str = Field(description="Concept name, exactly as listed.")
    cardinality: Literal["1:1", "1:N", "N:1", "N:M"] = Field(description="From the domain side: 1:N means one domain instance relates to many range instances.")
    evidence: str = Field(description="One sentence copied verbatim from the chunk that states this relation.")
    confidence: float = Field(ge=0.0, le=1.0)


class RelationResult(BaseModel):
    relations: list[RelationProposal]


PROPOSE_SYSTEM = """You infer non-hierarchical relations between the given ontology concepts from one chunk of a business process document.
Rules:
- Use only the listed concept names for domain and range.
- Only propose a relation that a sentence in this chunk states or directly implies; copy that sentence verbatim as evidence.
- Do not output is-a / subclass relations.
- Read the relation from domain to range and set cardinality from the domain side. If the text says the multiplicity is not established, choose N:M and lower the confidence.
- confidence reflects how explicitly the sentence supports the relation.
Return an empty list if the chunk states no relation between listed concepts."""

concept_block = "\n".join(f"- {n}: {c['description']}" for n, c in concepts.items())

proposals = []
if REUSE_CACHE and PROPOSALS_PATH.exists():
    proposals = read_json(PROPOSALS_PATH)["proposals"]
    print("reused cached proposals")
for ch in ([] if proposals else chunks):
    prompt = f"Concepts:\n{concept_block}\n\nChunk {ch['chunk_id']}:\n<<<\n{ch['text']}\n>>>\n\nPropose relations."
    res, err = generate_json(prompt, RelationResult, PROPOSE_SYSTEM)
    rels = res.relations if res else []
    for r in rels:
        proposals.append({"chunk_id": ch["chunk_id"], **r.model_dump()})
    print(ch["chunk_id"], len(rels), err or "")

print("proposals:", len(proposals))

reused cached proposals
proposals: 41


## Gate 4 · L1 결정론 검사 (제안 한 건씩)

- ① domain/range ∈ concept → 아니면 p3 (`concept_missing`)
- ③ 근거 문장이 청크 원문에 그대로 있는가 → 아니면 p1 (`evidence_not_in_source`)
- ③ 근거 문장에 domain과 range 표기가 나오는가 → 아니면 p1 (`term_missing`)

In [4]:
rejected = []
l1_passed = []

for k, p in enumerate(proposals):
    rid = f"{p['domain']}.{p['name']}.{p['range']}@{p['chunk_id']}"
    missing = [x for x in (p["domain"], p["range"]) if x not in concepts]
    if missing:
        rejected.append(gate_record(rid, "p4", "concept_missing", "p3", missing=missing, proposal=p))
        continue
    if norm_text(p["evidence"]) not in norm_text(chunk_by_id[p["chunk_id"]]["text"]):
        rejected.append(gate_record(rid, "p4", "evidence_not_in_source", "p1", proposal=p))
        continue
    absent = [x for x in (p["domain"], p["range"]) if not mentions_concept(p["evidence"], x)]
    if absent:
        rejected.append(gate_record(rid, "p4", "term_missing", "p1", chunk_id=p["chunk_id"], absent=absent, proposal=p))
        continue
    l1_passed.append(p)

print("L1 passed:", len(l1_passed), "| rejected:", Counter(r["reason"] for r in rejected))

L1 passed: 28 | rejected: Counter({'term_missing': 13})


## 병합과 ② 다중성 검사

같은 `(domain, name, range)`는 하나의 Relation Type으로 묶고 근거를 모은다. 청크마다 다중성이 다르면 사람 큐로 보낸다. 같은 concept 쌍의 역방향 관계는 표시만 한다(역관계일 수 있다).

In [5]:
grouped = defaultdict(list)
for p in l1_passed:
    grouped[(p["domain"], p["name"], p["range"])].append(p)

human_queue = []
merged = []
for (d, n, r), ps in grouped.items():
    rid = f"{d}.{n}.{r}"
    cards = Counter(p["cardinality"] for p in ps)
    rel = {
        "relation_id": rid,
        "name": n,
        "domain": d,
        "range": r,
        "cardinality": cards.most_common(1)[0][0],
        "gen_confidence": max(p["confidence"] for p in ps),
        "evidence": [{"chunk_id": p["chunk_id"], "sentence": p["evidence"]} for p in ps],
    }
    if len(cards) > 1:
        human_queue.append(gate_record(rid, "p4", "cardinality_conflict", "human", cardinalities=dict(cards), relation=rel))
        continue
    merged.append(rel)

pairs = defaultdict(list)
for rel in merged:
    pairs[frozenset((rel["domain"], rel["range"]))].append(rel["relation_id"])
for rel in merged:
    rel["same_pair_relations"] = [x for x in pairs[frozenset((rel["domain"], rel["range"]))] if x != rel["relation_id"]]

print("merged relation types:", len(merged), "| cardinality conflicts:", len(human_queue))

merged relation types: 28 | cardinality conflicts: 0


## Gate 4 · L3 LLM judge (relation 한 건씩, 새 컨텍스트)

- `verdict`: `supported` / `reversed`(근거는 반대 방향을 말함) / `unsupported`
- `cardinality_ok`: 근거가 제안된 다중성과 모순되지 않는가
- ④ `min(gen_confidence, judge confidence) < CONF_THRESHOLD` → 보류

In [6]:
class RelationJudgement(BaseModel):
    verdict: Literal["supported", "reversed", "unsupported"]
    cardinality_ok: bool
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str


JUDGE_SYSTEM = """You verify one proposed ontology relation against its evidence sentences only.
verdict=supported if the sentences state the relation in the given direction (domain -> range).
verdict=reversed if the sentences support the relation only with domain and range swapped.
verdict=unsupported if the sentences do not state it.
cardinality_ok=false if the sentences contradict the proposed multiplicity (read from the domain side), or say the multiplicity is not established while a strict one is proposed."""


def judge_relation(rel: dict) -> tuple[RelationJudgement | None, str | None]:
    prompt = (
        f"Relation: {rel['domain']} --{rel['name']}--> {rel['range']}  (cardinality {rel['cardinality']})\n"
        f"{rel['domain']}: {concepts[rel['domain']]['description']}\n"
        f"{rel['range']}: {concepts[rel['range']]['description']}\n"
        "Evidence sentences:\n" + "\n".join(f"  - {e['sentence']}" for e in rel["evidence"]) + "\n\nVerify the relation."
    )
    return generate_json(prompt, RelationJudgement, JUDGE_SYSTEM)


def judge_key(rel: dict) -> str:
    return f"{rel['relation_id']}|{rel['cardinality']}|" + "||".join(sorted(e["sentence"] for e in rel["evidence"]))


judge_cache = read_json(JUDGE_CACHE_PATH) if REUSE_CACHE and JUDGE_CACHE_PATH.exists() else {}
n_cached = 0

relations = []
for rel in merged:
    key = judge_key(rel)
    if key in judge_cache:
        j, err = RelationJudgement(**judge_cache[key]), None
        n_cached += 1
    else:
        j, err = judge_relation(rel)
        if j is not None:
            judge_cache[key] = j.model_dump()
    if j is None:
        human_queue.append(gate_record(rel["relation_id"], "p4", "judge_failed", "human", error=err, relation=rel))
        continue
    rel["judge"] = j.model_dump()
    rel["confidence"] = round(min(rel["gen_confidence"], j.confidence), 3)
    if j.verdict == "reversed":
        rejected.append(gate_record(rel["relation_id"], "p4", "direction_reversed", "p4", relation=rel))
    elif j.verdict == "unsupported":
        rejected.append(gate_record(rel["relation_id"], "p4", "unsupported_by_evidence", "p4", relation=rel))
    elif not j.cardinality_ok:
        human_queue.append(gate_record(rel["relation_id"], "p4", "cardinality_disputed", "human", relation=rel))
    elif rel["confidence"] < CONF_THRESHOLD:
        human_queue.append(gate_record(rel["relation_id"], "p4", "low_confidence", "human", relation=rel))
    else:
        relations.append(rel)

write_json(JUDGE_CACHE_PATH, judge_cache)
print("judge cache hits:", n_cached, "/", len(merged))
print("passed:", len(relations), "| rejected:", Counter(r["reason"] for r in rejected), "| held:", Counter(h["reason"] for h in human_queue))

judge cache hits: 27 / 28
passed: 25 | rejected: Counter({'term_missing': 13, 'unsupported_by_evidence': 2}) | held: Counter({'cardinality_disputed': 1})


## 결과 확인과 저장

In [7]:
for rel in sorted(relations, key=lambda r: (r["domain"], r["range"])):
    print(f"{rel['domain']:>20} --{rel['name']}--> {rel['range']:<22} {rel['cardinality']}  conf={rel['confidence']}  ev={len(rel['evidence'])}")

print("\nheld:")
for h in human_queue:
    print(f"  {h['item_id']:<55} {h['reason']}")

used = {c for rel in relations for c in (rel["domain"], rel["range"])}
print(f"\nconcepts covered by a relation: {len(used)}/{len(concepts)}  (isolated: {sorted(set(concepts) - used)})")

        BusinessRule --requiresActivity--> ProcessActivity        1:N  conf=0.9  ev=1
                Case --organizesEventLog--> EventLog               N:1  conf=0.8  ev=1
                Case --hasGoodsReceipt--> GoodsReceipt           1:N  conf=0.8  ev=1
                Case --hasInvoice--> Invoice                1:N  conf=0.8  ev=1
                Case --containsEvent--> RecordedEvent          1:N  conf=0.9  ev=1
             Company --recordsInvoice--> Invoice                1:N  conf=0.8  ev=1
        GoodsReceipt --relatesTo--> PurchaseOrderItem      N:1  conf=0.9  ev=1
             Invoice --covers--> PurchaseOrderItem      1:N  conf=0.9  ev=1
             Invoice --relatesTo--> PurchaseOrderItem      N:1  conf=0.9  ev=1
     ProcessActivity --occursInCase--> Case                   N:1  conf=0.8  ev=1
     ProcessActivity --applicableTo--> ProcessVariant         N:M  conf=0.85  ev=1
       PurchaseOrder --containsItem--> PurchaseOrderItem      1:N  conf=1.0  ev=1
       Purchas

In [8]:
meta = {
    "model": GEMINI_MODEL,
    "conf_threshold": CONF_THRESHOLD,
    "n_concepts_in": len(concepts),
    "n_proposals": len(proposals),
    "n_l1_passed": len(l1_passed),
    "n_relation_types_merged": len(merged),
    "n_relations": len(relations),
    "n_rejected": len(rejected),
    "n_human_queue": len(human_queue),
    "rejected_by_route": dict(Counter(r["route_to"] for r in rejected)),
    "concepts_covered": len(used),
}
write_json(OUTPUT_DIR / "relations.json", {"meta": meta, "relations": relations})
write_json(OUTPUT_DIR / "rejected.json", {"rejected": rejected})
write_json(OUTPUT_DIR / "human_queue.json", {"human_queue": human_queue})
write_json(OUTPUT_DIR / "proposals.json", {"proposals": proposals})
meta

{'model': 'gemini-3.5-flash',
 'conf_threshold': 0.7,
 'n_concepts_in': 17,
 'n_proposals': 41,
 'n_l1_passed': 28,
 'n_relation_types_merged': 28,
 'n_relations': 25,
 'n_rejected': 15,
 'n_human_queue': 1,
 'rejected_by_route': {'p1': 13, 'p4': 2},
 'concepts_covered': 13}